# Colab recognizer — Qwen2.5-VL-7B (4-bit) for the Khmer recognition A/B

Third contender in the recognizer A/B (Surya vs Tesseract-khm vs this). Runs the
**open-weights** Qwen2.5-VL-7B-Instruct in 4-bit on a free Colab **T4** (~4.5 GB weights;
dynamic resolution keeps Khmer coeng/subscripts legible on dense tables). Emits
`predictions.json` = `{"<image_file.png>": "<recognized text>"}`, which you score locally
**identically to the local engines**:

```bash
uv run python scripts/eval_recognizers.py --predictions predictions.json --name qwen2.5-vl-7b
```

**Steps:** Runtime ▸ Change runtime type ▸ **T4 GPU** → run all cells → when prompted, upload
the page PNGs from `eval/datasets/real/` (the same files the local A/B scores) → the last cell
downloads `predictions.json`.

Privacy: these are the financial PNGs; uploading to Colab is the open-model path chosen over a
commercial cloud API.

In [ ]:
!pip install -q -U "transformers>=4.49" accelerate bitsandbytes qwen-vl-utils pillow

In [ ]:
# Upload the page PNGs from eval/datasets/real/ (3 ARDB pages + the text page).
# Filenames are kept as-is and become the prediction keys, so they match the local GT.
from google.colab import files
uploaded = files.upload()
import glob
print(sorted(glob.glob('*.png')))

In [ ]:
import torch
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig

MODEL_ID = "Qwen/Qwen2.5-VL-7B-Instruct"
bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_ID, quantization_config=bnb, device_map="auto", torch_dtype=torch.bfloat16
)
# Bound resolution for T4 memory while keeping tiny Khmer subscripts legible.
# Raise max_pixels if you have Pro (L4/A100) and want sharper dense tables.
MIN_PIXELS = 256 * 28 * 28
MAX_PIXELS = 1280 * 28 * 28
processor = AutoProcessor.from_pretrained(MODEL_ID, min_pixels=MIN_PIXELS, max_pixels=MAX_PIXELS)
print("loaded", MODEL_ID)

In [ ]:
import json, glob
from qwen_vl_utils import process_vision_info

PROMPT = (
    "Transcribe ALL text in this image exactly as it appears, in natural reading order. "
    "The text is Khmer (ភាសាខ្មែរ) and may include numbers and tables. "
    "Output only the transcribed text — use Markdown table syntax for tables. "
    "Do not translate, summarize, or add any commentary."
)

preds = {}
for path in sorted(glob.glob("*.png")):
    messages = [{"role": "user", "content": [
        {"type": "image", "image": path},
        {"type": "text", "text": PROMPT},
    ]}]
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = processor(text=[text], images=image_inputs, videos=video_inputs,
                       padding=True, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=2048, do_sample=False)
    trimmed = out[:, inputs.input_ids.shape[1]:]
    result = processor.batch_decode(trimmed, skip_special_tokens=True,
                                    clean_up_tokenization_spaces=False)[0]
    preds[path] = result
    print(f"{path}: {len(result)} chars")

with open("predictions.json", "w", encoding="utf-8") as f:
    json.dump(preds, f, ensure_ascii=False, indent=2)
print("wrote predictions.json")

In [ ]:
from google.colab import files
files.download("predictions.json")